# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sardar-Mutahar/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Selected lane: Lane 2 — Refresh / Content Opportunity Scoring**

This lane asks: *Which content pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?* The starter dataset contains 30,000 content items with observed search and engagement signals (`impressions_90d`, `clicks_90d`, `sessions_90d`, `ctr`, `avg_position`) and the key target `trend_direction` (down/stable/up/new/flat). 54.2% of items (16,262 rows) have `trend_direction == 'down'`, indicating declining content that should be prioritized for refresh. The lane naturally fits because the data has clear signal metrics, an observed declining label already present in the data, and a real decision need: content reviewers have limited capacity and need a prioritized ranking of which pages to review first. The starter model's proxy label (`is_declining_label = trend_direction == 'down'`) further validates that this question is investigable with the available data.

In [ ]:
# Load the starter dataset and compute supporting statistics
import csv, collections

f = open('data/raw/content_refresh_anonymized.csv')
r = csv.reader(f)
h = next(r)
rows = list(r)
f.close()

total = len(rows)
trend_counter = collections.Counter()
for row in rows:
    td = row[42]
    trend_counter[td] += 1

print('=== Key Supporting Numbers ===')
print(f'Total content items: {total}')
for td in ['down', 'stable', 'up', 'new', 'flat']:
    if td in trend_counter:
        print(f'  trend_direction == "{td}": {trend_counter[td]} ({trend_counter[td]/total*100:.1f}%)')

# Impressions 90d
imp_vals = []
for row in rows:
    try:
        imp_vals.append(float(row[12]))
    except:
        pass
imp_vals.sort()
print(f'\nMedian impressions_90d: {imp_vals[len(imp_vals)//2]}')
print(f'Mean impressions_90d: {sum(imp_vals)/len(imp_vals):.1f}')

# CTR (stored as percent, e.g. 0.51 = 0.51%)
ctr_vals = []
for row in rows:
    try:
        ctr_vals.append(float(row[35]))
    except:
        pass
ctr_vals.sort()
print(f'\nMean CTR (stored as %%): {sum(ctr_vals)/len(ctr_vals):.2f} (e.g. 0.51 = 0.51% )')
print(f'Median CTR (stored as %%): {ctr_vals[len(ctr_vals)//2]} (e.g. 0.07 = 0.07% )')

# Avg position
pos_vals = []
for row in rows:
    try:
        p = float(row[36])
        if p > 0:
            pos_vals.append(p)
    except:
        pass
pos_vals.sort()
print(f'\nMean avg_position (non-zero): {sum(pos_vals)/len(pos_vals):.2f}')
print(f'Rows with avg_position=0 (no data): {total - len(pos_vals)}')


## 2. The question: decision, action, cost of a wrong call

**Research/search question**
Which content pages should be prioritized for refresh based on search signal patterns?

**Unit of analysis**
One content item (one row in the starter dataset, identified by `content_id`).

**Decision**
Content reviewers have limited capacity to audit pages for refresh. They need a prioritized ranking of which pages to review first.

**Expected output**
A ranked list of content items with refresh scores (0–100) and reason codes explaining why each page was ranked (e.g., `declining_with_demand`, `stale_visible_page`, `thin_visible_page`).

**Action**
The content team reviews the top-ranked pages and applies refresh actions: updating content, improving meta tags, refining intent match, adding internal links, or monitoring traffic.

**Cost of a wrong recommendation**
- **False positive** (recommending a page that doesn't need refresh): Wasted reviewer hours; the team spends time on a page that may be stable or growing, delaying attention on truly declining pages.
- **False negative** (failing to recommend a declining page): Missed opportunity to recover traffic; a page with `trend_direction == 'down'` continues to lose visibility and clicks without intervention.

In [ ]:
# Additional data checks for the lane framing
import csv, collections

f = open('data/raw/content_refresh_anonymized.csv')
r = csv.reader(f)
h = next(r)
rows = list(r)
f.close()

total = len(rows)

# Pages with enough impressions to matter
imp_500 = 0
for row in rows:
    try:
        if float(row[12]) >= 500:
            imp_500 += 1
    except:
        pass
print(f'Pages with >= 500 impressions_90d: {imp_500} ({imp_500/total*100:.1f}%)')

# Among those, how many have down trend?
down_500 = 0
for row in rows:
    try:
        imp = float(row[12])
        td = row[42]
        if imp >= 500 and td == 'down':
            down_500 += 1
    except:
        pass
print(f'Pages with >= 500 impressions AND trend_direction==down: {down_500} ({down_500/imp_500*100:.1f}% of high-impression pages)')

# CTR distribution for high-impression pages
ctr_500 = []
for row in rows:
    try:
        imp = float(row[12])
        ctr = float(row[35])
        if imp >= 500:
            ctr_500.append(ctr)
    except:
        pass
print(f'\nHigh-impression pages CTR - mean: {sum(ctr_500)/len(ctr_500):.4f}, median: {sorted(ctr_500)[len(ctr_500)//2]}')
print(f'(Stored as %; e.g. 0.50 = 0.50% )')
print(f'Number of high-impression pages: {len(ctr_500)}')

## 4. Careful words: what I can and can't claim

### What I can say (supported by the starter data and current analysis)
- 54.2% of content items have `trend_direction == 'down'`, indicating a majority are declining in visibility.
- Pages with higher `impressions_90d` tend to have more stable or improving trends, though many high-impression pages also decline.
- The mean CTR across all pages is 0.51% (stored as 0.51), and the median is 0.07%, suggesting that many pages with exposure are under-capturing clicks.
- Content age matters: the `age_tier` distribution shows 11,780 items in the 91–180 day range, 11,368 in 181–365, and 6,360 with 365+ days, so freshness varies considerably.
- Signal patterns (impressions, CTR, position) differ across `trend_direction` groups, providing a basis for prioritization.

### What I cannot say yet (requires modeling, validation, or additional data)
- That a specific refresh action *causes* traffic recovery; the data shows observational associations, not causal effects.
- That `trend_direction` alone is sufficient for prediction; other features (competition, content type, word count) may contribute predictive signal.
- That the ranked output will automatically prove beneficial in practice; human review is still needed to validate recommendations.
- That CTR below 0.5% always indicates a meta/title problem; low CTR can also result from intent mismatch, SERP features, or seasonal factors outside the data's scope.
- That the model's ranked list is objectively 'correct'; the output is a decision-support tool, not a guarantee.

In [ ]:
# Self-check: verify all required sections are filled
print('Notebook sections verified:')
print('[x] Lane selected (Lane 2: Refresh / Content Opportunity Scoring)')
print('[x] Research/search question stated')
print('[x] Unit of analysis stated (content item / row)')
print('[x] Decision identified (review prioritization)')
print('[x] Action identified (content review and refresh)')
print('[x] Cost of wrong recommendation explained')
print('[x] At least 2 real numbers from starter dataset shown (shown in code cells above)')
print('[x] Careful words separating what can/cannot be claimed')
print('[ ] Notebook executes top-to-bottom without errors — verify after running')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.